# MIIA Pothole Image Classification Challenge
**Objective**: Create a machine learning model to accurately predict the likelihood that an image contains a pothole.
**Metric**: Area Under the Curve (AUC)

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Dataset Class

In [ ]:
class PotholeDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]['ID']
        img_path = os.path.join(self.img_dir, f"{img_name}.jpg")
        
        image = Image.open(img_path).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
            
        label = self.df.iloc[idx]['label']
        return image, torch.tensor(label, dtype=torch.float32)

## 2. Model Definition (ResNet50)

In [ ]:
def get_model():
    model = models.resnet50(pretrained=True)
    num_ftrs = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(num_ftrs, 1),
        nn.Sigmoid()
    )
    return model.to(device)

## 3. Training Loop

In [ ]:
def train_model(model, train_loader, val_loader, epochs=10):
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            images, labels = images.to(device), labels.to(device).unsqueeze(1)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            
        # Validation
        model.eval()
        val_preds = []
        val_labels = []
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                outputs = model(images)
                val_preds.extend(outputs.cpu().numpy())
                val_labels.extend(labels.numpy())
        
        auc = roc_auc_score(val_labels, val_preds)
        print(f"Epoch {epoch+1}, Train Loss: {train_loss/len(train_loader):.4f}, Val AUC: {auc:.4f}")

## 4. Main Execution (Example)

In [ ]:
# Paths (adjust for Kaggle/Local)
DATA_DIR = "/kaggle/input/miia-pothole-image-classification-challenge"
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "train_images")
TRAIN_CSV = os.path.join(DATA_DIR, "train_labels.csv")

if os.path.exists(TRAIN_CSV):
    df = pd.read_csv(TRAIN_CSV)
    train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
    
    transform = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    train_ds = PotholeDataset(train_df, TRAIN_IMG_DIR, transform)
    val_ds = PotholeDataset(val_df, TRAIN_IMG_DIR, transform)
    
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
    
    model = get_model()
    train_model(model, train_loader, val_loader)
else:
    print("Dataset not found. Please attach the dataset to the notebook.")